# 18. 🧰 Прикладное: временные ряды и прогнозирование

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/18_time_series.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m18). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**18.1.** Сгенерируйте AR(1) с $\varphi = 0.8$, оцените $\varphi$ регрессией $y_t$ на $y_{t-1}$.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
y = np.zeros(5000)
for t in range(1, 5000):
    y[t] = 0.8 * y[t - 1] + rng.normal()
phi = np.sum(y[1:] * y[:-1]) / np.sum(y[:-1] ** 2)   # МНК без свободного члена
print(round(phi, 3))                                   # ≈ 0.8

### 🏋️ Практика модуля

**18.2.** Сравните сезонный наивный прогноз и скользящее среднее на почасовом ряде с суточной сезонностью по MAE.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
t = np.arange(24 * 60)
y = 10 + 3 * np.sin(2 * np.pi * t / 24) + rng.normal(0, 0.5, t.size)
test = slice(24 * 50, None)
naive_season = y[24 * 50 - 24:-24]                    # значение сутки назад
moving_avg = np.convolve(y, np.ones(24) / 24, mode="full")[24 * 50 - 1:-24]
print(np.abs(y[test] - naive_season).mean(), np.abs(y[test] - moving_avg).mean())
# сезонный наивный ≈ 0.53, скользящее среднее ≈ 1.94 — среднее «стирает» сезонность

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def time_series():
    rng = np.random.default_rng(0)
    n = 24 * 7 * 6
    t = np.arange(n)
    trend = 0.02 * t
    season = 3 * np.sin(2 * np.pi * t / 24) + 1.5 * np.sin(2 * np.pi * t / (24 * 7))
    noise = np.zeros(n)
    for i in range(1, n):
        noise[i] = 0.7 * noise[i - 1] + rng.normal(0, 0.8)      # AR(1)-шум
    y = 10 + trend + season + noise
    fig = plt.figure(figsize=(15, 6))
    gs = fig.add_gridspec(4, 2, width_ratios=[2.2, 1])
    ax0 = None
    for k, (comp, lab, c) in enumerate([(y, "ряд", "k"), (10 + trend, "тренд", C[0]),
                                        (season, "сезонность", C[2]), (noise, "остаток", C[1])]):
        ax = fig.add_subplot(gs[k, 0], sharex=ax0); ax0 = ax0 or ax
        ax.plot(t[:24 * 21], comp[:24 * 21], color=c, lw=1); ax.set_ylabel(lab)
        if k < 3: ax.tick_params(labelbottom=False)
    ax.set_xlabel("часы (первые 3 недели)")
    ax = fig.add_subplot(gs[:, 1])
    yd = y - np.polyval(np.polyfit(t, y, 1), t)              # убираем линейный тренд
    yc = yd - yd.mean(); lags = np.arange(0, 24 * 8)
    acf = np.array([np.sum(yc[:n - l] * yc[l:]) / np.sum(yc**2) for l in lags])
    ax.bar(lags, acf, width=1, color=C[0])
    ax.axhline(1.96 / np.sqrt(n), color=C[1], ls="--", lw=1); ax.axhline(-1.96 / np.sqrt(n), color=C[1], ls="--", lw=1)
    for d in range(1, 8): ax.axvline(24 * d, color="gray", lw=0.5, ls=":")
    ax.set_xlabel("лаг, часы"); ax.set_title("ACF после удаления тренда:\nпики каждые 24 часа — суточная сезонность")
    fig.suptitle("Временной ряд почасовой нагрузки (синтетика)", y=1.01)
    save(fig, "time_series")

time_series()

In [ ]:
def ts_cv():
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.4))
    for ax, kind in zip(axes, ["random", "time"]):
        rng = np.random.default_rng(1)
        for fold in range(5):
            if kind == "random":
                idx = rng.permutation(20); test = set(idx[fold * 4:(fold + 1) * 4]); train = set(range(20)) - test
            else:
                end = 8 + fold * 2; train = set(range(end)); test = set(range(end, end + 2))
            for i in range(20):
                col = C[0] if i in train else C[1] if i in test else "#e5e7eb"
                ax.add_patch(plt.Rectangle((i, 4 - fold), 0.9, 0.8, color=col))
        ax.set_xlim(0, 20); ax.set_ylim(-0.2, 5); ax.set_yticks([0.4 + k for k in range(5)], [f"fold {5 - k}" for k in range(5)])
        ax.set_xlabel("время →"); ax.grid(False)
        ax.set_title("✗ Случайный K-fold: модель «видит будущее»" if kind == "random" else "✓ Expanding window: train строго до test")
    axes[1].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=C[0]), plt.Rectangle((0, 0), 1, 1, color=C[1])],
                   labels=["train", "test"], loc="lower right")
    save(fig, "ts_cv")

ts_cv()